In [5]:
import sys
sys.path.append('C:/Users/wg984/Wolfgang/repos/ICU-Sleep/code1/')
sys.path.append('C:/Users/wg984/Wolfgang/repos/Bedmaster-ICU-tools/code/')
import numpy as np
import pandas as pd
from bmresearch_tools import BMR_load, BMR_plot, get_metadata 
from resample_BMR import remove_non_monotonic_data
%matplotlib widget
import matplotlib.pyplot as plt
import os

In [125]:
for studyid in range(8,33):
    studyid = str(studyid).zfill(3)
    os.mkdir('C:/Users/wg984/Dropbox (Partners HealthCare)/ICU-SLEEP-MATERIALS/TimeAlignmentAirGoBedmaster/Plots/'+studyid)

In [120]:
doc_path = 'C:/Users/wg984/Dropbox (Partners HealthCare)/ICU-SLEEP-MATERIALS/TimeAlignmentAirGoBedmaster/timealignment.csv'
doc = pd.read_csv(doc_path)
doc['study_id'] = doc['study_id'].apply(lambda x: str(x).zfill(3))
doc.head()

,study_id,airgo_available,bmr_available,code_successful,note,TS_manual,TS_auto1,TS_auto2,TS_auto3,person_to_check


In [121]:
doc_path = 'C:/Users/wg984/Dropbox (Partners HealthCare)/ICU-SLEEP-MATERIALS/TimeAlignmentAirGoBedmaster/timealignment.csv'
doc = pd.read_csv(doc_path)
doc['study_id'] = [str(x).zfill(3) for x in range(1,190)]
doc.to_csv(doc_path, index=False)

In [6]:
study_id = '001'
airgo_features_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/AirGo/airgo_features_v1/'
bmr_studyid_dir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_studyID/'
airgo_path = os.path.join(airgo_features_dir, 'airgo_'+study_id+'.csv')
bmr_path = os.path.join(bmr_studyid_dir, 'BMR_'+study_id+'.h5')

In [7]:
airgo_data = pd.read_csv(airgo_path)
if 'DateTime' in airgo_data.columns:
    airgo_data.rename(columns={'DateTime': 'datetime'}, inplace=True)
airgo_data.set_index('datetime', inplace=True)
airgo_data.index = pd.to_datetime(airgo_data.index, infer_datetime_format=1)

In [8]:
print(airgo_data.columns)
airgo_data.head()

Index(['band', 'accx', 'accy', 'accz', 'crcstatus', 'deriv1', 'deriv2',
       'ventilation0', 'ventilation_3s', 'ventilation_10s', 'ventilation_60s',
       'hypo_10_60', 'movavg_1s', 'movavg_1_2s', 'movavg_2s', 'movavg_3s',
       'movavg_5s', 'movavg_10s', 'movavg_60s', 'peaks', 'rr_10s', 'rr_60s'],
      dtype='object')


,band,accx,accy,accz,crcstatus,deriv1,deriv2,ventilation0,ventilation_3s,ventilation_10s,...,movavg_1s,movavg_1_2s,movavg_2s,movavg_3s,movavg_5s,movavg_10s,movavg_60s,peaks,rr_10s,rr_60s
datetime,,,,,,,,,,,,,,,,,,,,,
2018-06-06 15:26:00.400,16094.0,3.0,1.0,14.0,NaN,0.0,0.0,0.0,0.0,0.0,...,16094.000000,16094.000000,16094.000000,16094.000000,16094.000000,16094.000000,16094.000000,0,0.0,0.0
2018-06-06 15:26:00.500,16136.0,3.0,1.0,14.0,NaN,42.0,42.0,42.0,840.0,252.0,...,16115.000000,16115.000000,16115.000000,16115.000000,16115.000000,16115.000000,16115.000000,0,0.0,0.0
2018-06-06 15:26:00.600,16131.0,3.0,1.0,14.0,NaN,-5.0,-47.0,0.0,840.0,252.0,...,16120.333333,16120.333333,16120.333333,16120.333333,16120.333333,16120.333333,16120.333333,0,0.0,0.0
2018-06-06 15:26:00.700,16120.0,3.0,1.0,14.0,NaN,-11.0,-6.0,0.0,840.0,252.0,...,16120.250000,16120.250000,16120.250000,16120.250000,16120.250000,16120.250000,16120.250000,0,0.0,0.0
2018-06-06 15:26:00.800,16109.0,3.0,1.0,14.0,NaN,-11.0,0.0,0.0,840.0,252.0,...,16118.000000,16118.000000,16118.000000,16118.000000,16118.000000,16118.000000,16118.000000,0,0.0,0.0


In [9]:
airgo_data.tail()

,band,accx,accy,accz,crcstatus,deriv1,deriv2,ventilation0,ventilation_3s,ventilation_10s,...,movavg_1s,movavg_1_2s,movavg_2s,movavg_3s,movavg_5s,movavg_10s,movavg_60s,peaks,rr_10s,rr_60s
datetime,,,,,,,,,,,,,,,,,,,,,
2018-06-07 04:28:53.000,10104.000000,3.0,3.0,246.0,NaN,16.000000,17.250000,16.0,6560.0,3984.000000,...,10076.800000,10070.656250,10100.009375,10062.269410,9990.560000,9894.765379,9866.853741,1,18.0,15.0
2018-06-07 04:28:53.100,10085.333333,3.0,3.0,246.0,NaN,-18.666667,-34.666667,0.0,6560.0,3984.000000,...,10082.933333,10073.684028,10096.951042,10066.646914,9997.336354,9898.013333,9867.050511,0,18.0,15.0
2018-06-07 04:28:53.200,10050.666667,3.0,3.0,246.0,NaN,-34.666667,-16.000000,0.0,6560.0,3984.000000,...,10084.475000,10075.333333,10090.800000,10069.995062,10003.277188,9900.920000,9867.206648,0,18.0,15.0
2018-06-07 04:28:53.300,10032.000000,3.0,3.0,246.0,NaN,-18.666667,16.000000,0.0,6560.0,3976.303207,...,10081.675000,10076.000000,10083.200000,10072.827435,10008.623750,9903.627172,9867.348666,0,18.0,15.0
2018-06-07 04:28:53.400,10081.600000,3.0,3.0,246.0,NaN,49.600000,68.266667,49.6,7552.0,4252.352187,...,10081.360000,10079.862500,10078.423750,10077.386667,10014.675750,9906.794426,9867.590137,0,18.0,15.0


In [10]:
spo2_data = BMR_load(bmr_path, signals=['SPO2%'], loadEvents=0)
spo2_data = remove_non_monotonic_data(spo2_data)
spo2_data = spo2_data['SPO2%']
spo2_data.set_index('datetime', inplace=True)
spo2_data.drop('posix',axis=1, inplace=True)
spo2_data.rename(columns={'signal': 'spo2'}, inplace=True)

In [11]:
spo2_data.head()

,spo2
datetime,
2018-06-05 07:17:34,98.0
2018-06-05 07:17:36,98.0
2018-06-05 07:17:38,98.0
2018-06-05 07:17:40,98.0
2018-06-05 07:17:42,97.0


In [12]:
# resample spo2 and airgo between airgo min and airgo max.

In [13]:
spo2_data = spo2_data.loc[airgo_data.index[0] : airgo_data.index[-1],:]

In [14]:
data = pd.concat([spo2_data, airgo_data], join='outer', axis=1, sort=True)

In [15]:
data.spo2 = data.spo2.interpolate(method='pchip', order=3, limit_area='inside',
                          limit=None) 
data.shape

(469731, 23)

In [16]:
data = data.loc[(1- np.isnan(data.spo2.values)).astype('bool'),:]
data.shape

(469721, 23)

In [17]:
data

,spo2,band,accx,accy,accz,crcstatus,deriv1,deriv2,ventilation0,ventilation_3s,...,movavg_1s,movavg_1_2s,movavg_2s,movavg_3s,movavg_5s,movavg_10s,movavg_60s,peaks,rr_10s,rr_60s
datetime,,,,,,,,,,,,,,,,,,,,,
2018-06-06 15:26:01.000,96.0,16158.645161,3.000000,1.000000,14.000000,NaN,54.645161,59.645161,54.645161,1932.903226,...,16121.806452,16121.806452,16121.806452,16121.806452,16121.806452,16121.806452,16121.806452,0,0.0,0.0
2018-06-06 15:26:01.100,96.0,16216.000000,3.000000,1.000000,14.000000,NaN,57.354839,2.709677,57.354839,3080.000000,...,16133.580645,16133.580645,16133.580645,16133.580645,16133.580645,16133.580645,16133.580645,0,0.0,0.0
2018-06-06 15:26:01.200,96.0,16219.390307,2.740741,0.740741,14.259259,NaN,3.390307,-53.964532,3.390307,3147.806135,...,16143.115052,16143.115052,16143.115052,16143.115052,16143.115052,16143.115052,16143.115052,0,0.0,0.0
2018-06-06 15:26:01.300,96.0,16220.945488,2.259259,0.259259,14.740741,NaN,1.555181,-1.835125,1.555181,3178.909761,...,16150.898096,16150.898096,16150.898096,16150.898096,16150.898096,16150.898096,16150.898096,0,0.0,0.0
2018-06-06 15:26:01.400,96.0,16224.000000,2.000000,0.000000,15.000000,NaN,3.054512,1.499331,3.054512,3240.000000,...,16163.898096,16157.543723,16157.543723,16157.543723,16157.543723,16157.543723,16157.543723,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-06-07 04:28:52.600,95.0,10094.750000,3.000000,3.000000,245.156250,NaN,-1.250000,-12.500000,0.000000,6240.000000,...,10065.325000,10078.802083,10090.021875,10044.891667,9962.768600,9881.701042,9866.186205,0,12.0,14.0
2018-06-07 04:28:52.700,95.0,10092.000000,3.000000,3.000000,245.500000,NaN,-2.750000,-1.500000,0.000000,6240.000000,...,10064.125000,10073.218750,10095.546875,10049.052435,9969.631400,9884.899687,9866.324991,0,12.0,14.0
2018-06-07 04:28:52.800,95.0,10089.250000,3.000000,3.000000,245.843750,NaN,-2.750000,0.000000,0.000000,6240.000000,...,10065.587500,10069.541667,10099.012500,10053.195062,9976.491600,9888.112420,9866.476442,0,12.0,14.0


In [37]:
data['hypo_10_60_movavg2sec'] = data.hypo_10_60.rolling('2s').mean()
data['hypo_10_60_movavg4sec'] = data.hypo_10_60.rolling('4s').mean()
data['hypo_10_60_movavg6sec'] = data.hypo_10_60.rolling('6s').mean()
data['hypo_10_60_movavg8sec'] = data.hypo_10_60.rolling('8s').mean()
data['hypo_10_60_movavg10sec'] = data.hypo_10_60.rolling('10s').mean()


In [19]:
data['hypo_10_60_movavg10sec'].values.shape

(469721,)

In [20]:
np.ones(data['hypo_10_60_movavg10sec'].shape).shape

(469721,)

In [38]:
data['hypo_10_60_movavg4sec'] = np.minimum(data['hypo_10_60_movavg4sec'].values,np.ones(data['hypo_10_60_movavg4sec'].shape)*1.2)
data['hypo_10_60_movavg6sec'] = np.minimum(data['hypo_10_60_movavg6sec'].values,np.ones(data['hypo_10_60_movavg6sec'].shape)*1.2)
data['hypo_10_60_movavg8sec'] = np.minimum(data['hypo_10_60_movavg8sec'].values,np.ones(data['hypo_10_60_movavg8sec'].shape)*1.2)
data['hypo_10_60_movavg10sec'] = np.minimum(data['hypo_10_60_movavg10sec'].values,np.ones(data['hypo_10_60_movavg10sec'].shape)*1.2)


In [22]:
plt.figure()
plt.plot(data.hypo_10_60.index, data.hypo_10_60)
plt.plot(data.hypo_10_60_movavg2sec.index, data.hypo_10_60_movavg2sec)
plt.plot(data.hypo_10_60_movavg4sec.index, data.hypo_10_60_movavg4sec)
plt.legend(['10-60', '2sec','4 sec'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

C:\Users\wg984\.conda\envs\visualization\lib\site-packages\pandas\plotting\_matplotlib\converter.py:103: FutureWarning:

Using an implicitly registered datetime converter for a matplotlib plotting method. The converter was registered by pandas on import. Future versions of pandas will require you to explicitly register matplotlib converters.

To register the converters:
	>>> from pandas.plotting import register_matplotlib_converters
	>>> register_matplotlib_converters()



In [23]:
plt.close()

In [24]:
idx_start = 386000
idx_end = 390000

idx_start = 380000
idx_end = 400000

In [25]:
plt.figure()
ax0= plt.subplot(311)
plt.plot(data.band.index[idx_start:idx_end], data.band[idx_start:idx_end])
ax1= plt.subplot(312, sharex=ax0)
plt.plot(data.hypo_10_60.index[idx_start:idx_end], data.hypo_10_60[idx_start:idx_end])
# plt.plot(data.hypo_10_60_movavg2sec.index[idx_start:idx_end], data.hypo_10_60_movavg2sec[idx_start:idx_end])
# plt.plot(data.hypo_10_60_movavg4sec.index[idx_start:idx_end], data.hypo_10_60_movavg4sec[idx_start:idx_end])
# plt.plot(data.hypo_10_60_movavg6sec.index[idx_start:idx_end], data.hypo_10_60_movavg6sec[idx_start:idx_end])
plt.plot(data.hypo_10_60_movavg8sec.index[idx_start:idx_end], data.hypo_10_60_movavg8sec[idx_start:idx_end])
# plt.plot(data.hypo_10_60_movavg10sec.index[idx_start:idx_end], data.hypo_10_60_movavg10sec[idx_start:idx_end])

ax2= plt.subplot(313, sharex=ax0)
plt.plot(data.spo2.index[idx_start:idx_end], data.spo2[idx_start:idx_end])


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [26]:
plt.figure()
ax0= plt.subplot(311)
plt.plot(data.band.index[idx_start:idx_end], data.band[idx_start:idx_end])
ax1= plt.subplot(312, sharex=ax0)
# plt.plot(data.hypo_10_60.index[idx_start:idx_end], data.hypo_10_60[idx_start:idx_end])
# plt.plot(data.hypo_10_60_movavg2sec.index[idx_start:idx_end], data.hypo_10_60_movavg2sec[idx_start:idx_end])
# plt.plot(data.hypo_10_60_movavg4sec.index[idx_start:idx_end], data.hypo_10_60_movavg4sec[idx_start:idx_end])
plt.plot(data.hypo_10_60_movavg6sec.index[idx_start:idx_end-1], np.diff(data.hypo_10_60_movavg6sec[idx_start:idx_end]))
plt.plot(data.hypo_10_60_movavg8sec.index[idx_start:idx_end-1], np.diff(data.hypo_10_60_movavg8sec[idx_start:idx_end]))
plt.plot(data.hypo_10_60_movavg10sec.index[idx_start:idx_end-1], np.diff(data.hypo_10_60_movavg10sec[idx_start:idx_end]))

ax2= plt.subplot(313, sharex=ax0)
plt.plot(data.spo2.index[idx_start:idx_end-1], np.diff(data.spo2[idx_start:idx_end]))


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [27]:
spearman = []
pearson = []
lagrange = range(-3000,3000,10)
for lag in lagrange:
#     hypo = data.hypo_10_60_movavg8sec.iloc[idx_start:idx_end]
#     spo2 = data.spo2.iloc[idx_start-lag:idx_end-lag]
    hypo = pd.Series(np.diff(data.hypo_10_60_movavg8sec.iloc[idx_start:idx_end]))
    spo2 = pd.Series(np.diff(data.spo2.iloc[idx_start-lag:idx_end-lag]))
    spearman.append(spo2.corr(hypo, method='spearman'))
    pearson.append(spo2.corr(hypo, method='pearson'))

plt.figure()
plt.plot(np.array(lagrange)/10, spearman)
plt.plot(np.array(lagrange)/10, pearson)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [28]:
for i in range(40):
    plt.close()

In [29]:
idx_start = 380000
idx_end = 400000
lagrange = range(-3000,3000,10)

maxspearman = []
maxpearson = []

hypo_diff = pd.Series(np.diff(data.hypo_10_60_movavg8sec)) 
spo2_diff = pd.Series(np.diff(data.spo2))

for lag in lagrange:

        
        spearman.append(spo2_diff.corr(hypo_diff, method='spearman'))
        pearson.append(spo2_diff.corr(hypo_diff, method='pearson'))
    

KeyboardInterrupt: 

In [ ]:

plt.close()
# plt.figure(figsize=(50,10))

idx_start = 380000
idx_end = 400000
lagrange = range(-3000,3000,10)

maxspearman = []
maxpearson = []

hypo_diff = pd.Series(np.diff(data.hypo_10_60_movavg8sec)) 
spo2_diff = pd.Series(np.diff(data.spo2))


spearman =[]
pearson = []

for lag in lagrange:
#         hypo = data.hypo_10_60_movavg2sec.iloc[idx_start:idx_start+20000]
#         spo2 = data.spo2.iloc[idx_start+lag:idx_start+20000+lag]
    hypo = hypo_diff.iloc[idx_start:idx_end]
    spo2 = spo2_diff.iloc[idx_start-lag:idx_end-lag]

    spearman.append(spo2.corr(hypo, method='spearman'))
    pearson.append(spo2.corr(hypo, method='pearson'))

maxspearman.append(lagrange[np.argmax(spearman)]/10)
maxpearson.append(lagrange[np.argmax(pearson)]/10)

#     print(lagrange[np.argmax(spearman)]/10)
#     print(lagrange[np.argmax(pearson)]/10)
#     print('\n')
#     plt.subplot(len(searchrange),1,i+1)
#     plt.plot(np.array(lagrange)/10,spearman)
#     plt.plot(np.array(lagrange)/10,pearson)

In [ ]:
maxpearson

In [ ]:
maxspearman

In [ ]:
data.shape[0]

In [47]:

plt.close()
# plt.figure(figsize=(50,10))

# idx_start = 380000
# idx_end = 400000
lagrange = range(-3000,3000,10)
searchrange = range(0,data.shape[0]-5001,5000)

maxspearman = []
maxpearson = []

# hypo_diff = pd.Series(np.diff(data.hypo_10_60_movavg8sec)) 
# spo2_diff = pd.Series(np.diff(data.spo2))

# searchrange = range(0,spo2_diff.shape[0]-5000,5000)

for [i,idx_start] in enumerate(searchrange):
    spearman =[]
    pearson = []
    idx_end = idx_start + 20000
    
    for lag in lagrange:
#         hypo = data.hypo_10_60_movavg2sec.iloc[idx_start:idx_start+20000]
#         spo2 = data.spo2.iloc[idx_start+lag:idx_start+20000+lag]

#         hypo = hypo_diff.iloc[idx_start:idx_end]
#         spo2 = spo2_diff.iloc[idx_start-lag:idx_end-lag]

        hypo = pd.Series(np.diff(data.hypo_10_60_movavg8sec.iloc[idx_start:idx_end]))
        spo2 = pd.Series(np.diff(data.spo2.iloc[idx_start-lag:idx_end-lag]))
        
        spearman.append(spo2.corr(hypo, method='spearman'))
        pearson.append(spo2.corr(hypo, method='pearson'))
    
    maxspearman.append(lagrange[np.argmax(spearman)]/10)
    maxpearson.append(lagrange[np.argmax(pearson)]/10)
    
#     print(lagrange[np.argmax(spearman)]/10)
#     print(lagrange[np.argmax(pearson)]/10)
#     print('\n')
#     plt.subplot(len(searchrange),1,i+1)
#     plt.plot(np.array(lagrange)/10,spearman)
#     plt.plot(np.array(lagrange)/10,pearson)

In [44]:
idx_start = 380000
idx_end = 400000
plt.figure()
ax0= plt.subplot(211)
plt.plot(data.hypo_10_60_movavg8sec.iloc[idx_start:idx_end])
plt.legend(['hypopnea indication curve'])
ax1= plt.subplot(212, sharex=ax0)
plt.plot(data.spo2.iloc[idx_start:idx_end])
plt.legend(['SPO2%'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [46]:
idx_start = 380000
idx_end = 400000
plt.figure()
plt.plot(np.diff(data.hypo_10_60_movavg8sec.iloc[idx_start:idx_end]))
plt.plot(np.diff(data.spo2.iloc[idx_start:idx_end]))
plt.legend(['deriv. hypopnea indication curve','deriv SPO2%'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [69]:
# note: -6 beause of 8sec moving average. approximation to correct for it
plt.figure()
plt.hist(np.array(maxspearman),histtype='step', bins=len(lagrange)//2)
plt.hist(np.array(maxpearson),histtype='step', bins=len(lagrange)//2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([ 2.,  0.,  0.,  1.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  3.,  5.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  1.,  0.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  4.,  1.,  9., 17.,  5.,  4.,  2.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  2.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,
         0.,  0.,  0.,  1.,  3.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.,  1.,  1.,  0.,  0.,  0.,  0.,
         0.,  2.,  0.,  1.,  0.,  0.,  0.,  0.,  0.

In [49]:
np.argsort(maxspearman)

array([12, 33, 22, 10, 11,  9, 47, 30, 29, 31, 92, 24, 46, 36,  6, 90, 89,
       88, 91, 87, 68, 67, 66, 65, 64, 62, 63, 86, 61, 60, 83, 84, 85, 69,
       70, 59, 81, 72, 73, 74, 75, 76, 77, 80, 71, 82, 57, 56, 55, 54, 58,
       79, 78,  3, 49, 48, 51, 53, 50, 52, 13,  1, 15, 32,  7,  8, 16, 17,
       39, 38, 40,  0, 20, 18, 19, 41, 42, 44, 43, 45, 35, 14, 23,  2, 34,
       21, 28, 26, 25, 37,  4,  5, 27], dtype=int64)

In [76]:
from scipy.signal import argrelextrema, find_peaks

maxspearman = np.array(maxspearman)
# for local maxima
counts, lag = np.histogram(maxspearman, len(lagrange)//2)

In [79]:
# local_max = argrelextrema(counts, np.greater)
# local_max_lag = lag[local_max]

local_max = find_peaks(counts, distance=5)[0]
local_max_lag = lag[local_max]
local_max_lag


array([-244.28, -232.34, -220.4 , -190.55, -122.89, -108.96,  -81.1 ,
        -67.17,  -55.23,  -37.32,  -11.45,   -1.5 ,   20.39,   30.34,
         42.28,   58.2 ,   70.14,  107.95,  239.29,  251.23,  267.15,
        285.06])

In [85]:
local_max_count = counts[local_max]
print(local_max_count)
idx_candidates = np.argsort(local_max_count)[::-1][:3]
print(local_max_lag[idx_candidates])
print(local_max_count[idx_candidates])
# np.argsort()

[ 1  3  1  2  1  1  1 18  1  1  3  3  2  4  1  1  1  1  1  2  1  1]
[-67.17  30.34 -11.45]
[18  4  3]


In [104]:
doc.loc[doc.study_id==study_id,'TS_auto1'] = idx_candidates[0]
doc.loc[doc.study_id==study_id,'TS_auto2'] = idx_candidates[1]
doc.loc[doc.study_id==study_id,'TS_auto3'] = idx_candidates[2]

In [105]:
doc

,study_id,airgo_available,plot_done,note,TS_manual,TS_auto1,TS_auto2,TS_auto3,person_to_check
0,001,NaN,NaN,NaN,NaN,7,13,10,NaN
1,002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
184,185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
185,186,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,187,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,188,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [96]:
study_id

'001'

In [31]:
# note: -6 beause of 8sec moving average. approximation to correct for it
plt.figure()
plt.hist(np.array(maxspearman)-6,histtype='step', bins=len(lagrange)//2)
plt.hist(np.array(maxpearson)-6,histtype='step', bins=len(lagrange)//2)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(array([ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  4.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  4.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         1.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,
         0.,  0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  2.,  0.,  0.,
         0.,  0.,  0.,  0.,  0.,  4.,  2., 15., 11.,  3.,  2.,  4.,  0.,
         0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  0.,  0.,  0.,  3.,  1.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  1.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
         0.,  2.,  0.,  0.,  1.,  0.,  1.,  0.,  0.

In [ ]:

# plt.close()
# plt.figure(figsize=(50,10))

# idx_start = 380000
# idx_end = 400000
# lagrange = range(-3000,3000,10)
# searchrange = range(0,450000,20000)
# maxspearman = []
# maxpearson = []

# for [i,idx_start] in enumerate(searchrange):
#     spearman =[]
#     pearson = []
#     idx_end = idx_start + 20000
    
#     for lag in lagrange:
# #         hypo = data.hypo_10_60_movavg2sec.iloc[idx_start:idx_start+20000]
# #         spo2 = data.spo2.iloc[idx_start+lag:idx_start+20000+lag]
#         hypo = pd.Series(np.diff(data.hypo_10_60_movavg8sec.iloc[idx_start:idx_end]))
#         spo2 = pd.Series(np.diff(data.spo2.iloc[idx_start-lag:idx_end-lag]))
    
#         spearman.append(spo2.corr(hypo, method='spearman'))
#         pearson.append(spo2.corr(hypo, method='pearson'))
    
#     maxspearman.append(lagrange[np.argmax(spearman)]/10)
#     maxpearson.append(lagrange[np.argmax(pearson)]/10)
    
# #     print(lagrange[np.argmax(spearman)]/10)
# #     print(lagrange[np.argmax(pearson)]/10)
# #     print('\n')
#     plt.subplot(len(searchrange),1,i+1)
#     plt.plot(np.array(lagrange)/10,spearman)
#     plt.plot(np.array(lagrange)/10,pearson)

In [184]:

plt.close()
plt.figure(figsize=(50,10))

for [i,idx_start] in enumerate(range(40000,200000,20000)):
    spearman =[]
    pearson = []
    for lag in range(-600,600,10):
        hypo = data.hypo_10_60_movavg2sec.iloc[idx_start:idx_start+5000]
        spo2 = data.spo2.iloc[idx_start+lag:idx_start+5000+lag]
        spearman.append(spo2.corr(hypo, method='spearman'))
        pearson.append(spo2.corr(hypo, method='pearson'))
    plt.subplot(len(range(40000,200000,20000)),1,i+1)
    plt.plot(spearman)
#     plt.plot(pearson)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [181]:
plt.close()
plt.figure(figsize=(50,10))

plt.plot(spearman)
plt.plot(pearson)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [180]:
i

7

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

0.00626685049219263

datetime
2018-06-06 15:42:41.000    0.652884
2018-06-06 15:42:41.100    0.657893
2018-06-06 15:42:41.200    0.662863
2018-06-06 15:42:41.300    0.668361
2018-06-06 15:42:41.400    0.674664
                             ...   
2018-06-06 15:59:20.500    0.806488
2018-06-06 15:59:20.600    0.807903
2018-06-06 15:59:20.700    0.808276
2018-06-06 15:59:20.800    0.808435
2018-06-06 15:59:20.900    0.808447
Name: hypo_10_60_movavg2sec, Length: 10000, dtype: float64